# FAISS 임베딩 인덱스 구축

Hugging Face 데이터셋을 다운로드하여 마크다운 섹션별로 청킹하고, multilingual-e5-large-instruct 모델로 임베딩한 후 FAISS 인덱스로 저장합니다.


## 1. 환경 설정 및 라이브러리 임포트


In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
import logging
import importlib.util

# 프로젝트 루트 경로 추가
project_root = Path().resolve().parent.parent.parent
sys.path.insert(0, str(project_root))

# 설정 임포트
from src.config.config import (
    FAISS_INDEX_DIR,
    EMBEDDING_MODEL_NAME,
    EMBEDDING_MAX_TOKENS,
    CHUNKING_HEADERS,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
    EMBEDDING_BATCH_SIZE,
    EMBEDDING_DEVICE
)

# ke-02 모듈 임포트 (디렉토리 이름에 하이픈이 있어서 importlib 사용)
ke02_path = project_root / "scripts" / "experiments" / "ke-02"
sys.path.insert(0, str(ke02_path))

# 모듈 임포트
from markdown_chunker import (
    chunk_markdown_document,
    prepare_embedding_text,
    chunk_dataset
)
from embedder import create_embedder
from faiss_indexer import FAISSIndexer

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"Project root: {project_root}")
print(f"FAISS index directory: {FAISS_INDEX_DIR}")


Current working directory: /data/ephemeral/home/T8001/pro-nlp-generationfornlp-nlp-07
Project root: /data/ephemeral/home/T8001/pro-nlp-generationfornlp-nlp-07
Project root exists: True


Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu128 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


ModuleNotFoundError: No module named 'langchain.schema'

## 2. Hugging Face 데이터셋 다운로드


In [ ]:
dataset_name = "NLP-07-ODQA/kowiki-cleaned"

print(f"Loading dataset: {dataset_name}")
dataset = load_dataset(dataset_name)

# train split 사용 (또는 적절한 split 선택)
if 'train' in dataset:
    data = dataset['train']
elif 'train' not in dataset and len(dataset) > 0:
    # 첫 번째 split 사용
    split_name = list(dataset.keys())[0]
    data = dataset[split_name]
else:
    data = dataset

print(f"Dataset loaded. Total rows: {len(data)}")
print(f"Columns: {data.column_names}")

# 샘플 데이터 확인
if len(data) > 0:
    print("\nSample row:")
    sample = data[0]
    for key, value in sample.items():
        if isinstance(value, str) and len(value) > 100:
            print(f"{key}: {value[:100]}...")
        else:
            print(f"{key}: {value}")


## 3. 청킹 예시 확인 (샘플 문서)


In [ ]:
# 샘플 문서 선택 (1-2개)
sample_indices = [0, 1] if len(data) > 1 else [0]
sample_docs = [data[idx] for idx in sample_indices]

all_sample_chunks = []

for sample_doc in sample_docs:
    context = sample_doc.get('context', '')
    title = sample_doc.get('title', '')
    page_id = sample_doc.get('page_id', '')
    
    print(f"\n{'='*80}")
    print(f"Title: {title}")
    print(f"Page ID: {page_id}")
    print(f"Context length: {len(context)} characters")
    print(f"{'='*80}")
    
    # 청킹 실행
    chunks = chunk_markdown_document(
        context=context,
        title=title,
        page_id=page_id,
        headers_to_split_on=CHUNKING_HEADERS,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )
    
    all_sample_chunks.extend(chunks)
    
    print(f"\nNumber of chunks: {len(chunks)}")
    
    # 청킹 결과 시각화
    chunk_data = []
    for i, chunk in enumerate(chunks):
        header_path = []
        for level in ["Header 1", "Header 2", "Header 3"]:
            if level in chunk.metadata:
                header_path.append(chunk.metadata[level])
        
        header_str = " > ".join(header_path) if header_path else "(No header)"
        estimated_tokens = int(len(chunk.page_content) * 1.2)  # 대략적인 토큰 수
        
        chunk_data.append({
            'chunk_id': i,
            'title': chunk.metadata.get('title', ''),
            'page_id': chunk.metadata.get('page_id', ''),
            'header_path': header_str,
            'text_length': len(chunk.page_content),
            'estimated_tokens': estimated_tokens,
            'within_limit': estimated_tokens <= EMBEDDING_MAX_TOKENS,
            'preview': chunk.page_content[:100] + '...' if len(chunk.page_content) > 100 else chunk.page_content
        })
    
    # DataFrame으로 표시
    df_chunks = pd.DataFrame(chunk_data)
    print("\nChunking Results:")
    print(df_chunks.to_string(index=False))
    
    # 각 청크의 내용 미리보기
    print("\n" + "="*80)
    print("Chunk Details:")
    print("="*80)
    for i, chunk in enumerate(chunks[:3]):  # 처음 3개만 표시
        print(f"\n--- Chunk {i+1} ---")
        print(f"Metadata: {chunk.metadata}")
        print(f"Content (first 200 chars): {chunk.page_content[:200]}...")
        
        # 임베딩용 텍스트 생성
        embedding_text = prepare_embedding_text(chunk)
        print(f"\nEmbedding text (first 200 chars): {embedding_text[:200]}...")
        print(f"Embedding text length: {len(embedding_text)} characters")

print(f"\n\nTotal sample chunks: {len(all_sample_chunks)}")


## 4. 전체 데이터셋 청킹


In [ ]:
print("Starting chunking for entire dataset...")
print(f"Total documents: {len(data)}")

all_chunks = []

for idx, row in enumerate(tqdm(data, desc="Chunking documents")):
    context = row.get('context', '')
    title = row.get('title', '')
    page_id = row.get('page_id', '')
    
    if not context or not context.strip():
        continue
    
    chunks = chunk_markdown_document(
        context=context,
        title=title,
        page_id=page_id,
        headers_to_split_on=CHUNKING_HEADERS,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )
    
    all_chunks.extend(chunks)
    
    if (idx + 1) % 1000 == 0:
        print(f"Processed {idx + 1} documents, total chunks: {len(all_chunks)}")

print(f"\nChunking completed!")
print(f"Total documents: {len(data)}")
print(f"Total chunks: {len(all_chunks)}")
print(f"Average chunks per document: {len(all_chunks) / len(data):.2f}")


## 5. 임베딩 생성 (GPU 배치 처리)


In [ ]:
# Embedder 생성
print(f"Creating embedder with model: {EMBEDDING_MODEL_NAME}")
print(f"Device: {EMBEDDING_DEVICE}, Batch size: {EMBEDDING_BATCH_SIZE}")

embedder = create_embedder(
    model_name=EMBEDDING_MODEL_NAME,
    device=EMBEDDING_DEVICE,
    batch_size=EMBEDDING_BATCH_SIZE,
    max_length=EMBEDDING_MAX_TOKENS
)

print(f"Embedding dimension: {embedder.get_embedding_dim()}")


In [ ]:
# 임베딩용 텍스트 준비
print("Preparing embedding texts...")
embedding_texts = [prepare_embedding_text(chunk) for chunk in tqdm(all_chunks, desc="Preparing texts")]

print(f"Total texts to embed: {len(embedding_texts)}")
print(f"Sample text length: {len(embedding_texts[0]) if embedding_texts else 0} characters")


In [ ]:
# 임베딩 생성
print("Generating embeddings...")
embeddings = embedder.encode_texts(
    texts=embedding_texts,
    show_progress_bar=True
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")


## 6. FAISS 인덱스 구축 및 저장


In [ ]:
# FAISS 인덱서 생성
print("Creating FAISS index...")
indexer = FAISSIndexer(
    embedding_dim=embeddings.shape[1],
    index_type="cosine"  # 코사인 유사도 사용
)

# 문서와 임베딩 추가
print("Adding embeddings to index...")
indexer.add_documents(embeddings, all_chunks)

# 통계 정보
stats = indexer.get_stats()
print(f"\nIndex statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")


In [ ]:
# 인덱스 저장
FAISS_INDEX_DIR.mkdir(parents=True, exist_ok=True)

index_path = FAISS_INDEX_DIR / "faiss_index.bin"
metadata_path = FAISS_INDEX_DIR / "metadata.json"

print(f"Saving index to: {index_path}")
indexer.save(index_path, metadata_path)

print(f"\nIndex saved successfully!")
print(f"Index file: {index_path}")
print(f"Metadata file: {metadata_path}")


## 7. 인덱스 검증 (테스트 검색)


In [ ]:
# 테스트 쿼리로 검색
test_query = "한국의 역사"

print(f"Test query: {test_query}")

# 쿼리 임베딩 생성
query_text = f"query: {test_query}"  # e5 모델의 query 포맷
query_embedding = embedder.encode_texts([query_text], show_progress_bar=False)

# 검색
results = indexer.search(query_embedding[0], k=5)

print(f"\nTop 5 search results:")
print("="*80)
for i, result in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(f"  Distance: {result['distance']:.4f}")
    print(f"  Title: {result['metadata'].get('title', 'N/A')}")
    print(f"  Page ID: {result['metadata'].get('page_id', 'N/A')}")
    
    # Header 경로
    header_path = []
    for level in ["Header 1", "Header 2", "Header 3"]:
        if level in result['metadata']:
            header_path.append(result['metadata'][level])
    if header_path:
        print(f"  Header path: {' > '.join(header_path)}")
    
    content = result['metadata'].get('page_content', '')
    print(f"  Content preview: {content[:200]}...")
